In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy._lib.array_api_compat.numpy import linalg


# ---------------------------
# Helpers: Kronecker products
# ---------------------------

def kron3(A, B, C):
    """Sparse Kronecker product A ⊗ B ⊗ C."""
    return sp.kron(sp.kron(A, B, format="csr"), C, format="csr")

def kron3_vec(ax, ay, az):
    """Vector Kronecker product for 1D numpy arrays -> 1D numpy array."""
    return np.kron(np.kron(ax, ay), az)


# ---------------------------
# 1D operators: R and D4
# ---------------------------

def laplace_1d_dirichlet_matrix(N):
    """
    R: 1D discrete operator for (-d^2/dx^2) with Dirichlet BC eliminated.
    Grid: x_j=j*h, j=0..N, h=1/N
    Unknowns: u_1..u_{N-1}, size m=N-1
    """
    if N < 2:
        raise ValueError("N must be >= 2.")
    h = 1.0 / N
    m = N - 1
    main = 2.0 * np.ones(m)
    off  = -1.0 * np.ones(m - 1)
    R = sp.diags([off, main, off], [-1, 0, 1], format="csr") / (h * h)
    return R, h


def D4_1d_clamped_matrix_and_bases(N):
    """
    Closure-based 1D D4 approximating u'''' on interior unknowns u_1..u_{N-1}.
    Generally nonsymmetric.

    Also returns two boundary RHS basis vectors for derivative boundary data:
      b = b_du0 * du0 + b_du1 * du1,
    assuming u(0)=u(1)=0 along that 1D line.

    Returns:
      D4 : (m,m) csr sparse, scaled by 1/h^4
      b_du0, b_du1 : (m,) numpy arrays, scaled consistently
      h : float
    """
    if N < 5:
        raise ValueError("Need N >= 5 for this clamped closure stencil.")
    h = 1.0 / N
    m = N - 1
    D4 = sp.lil_matrix((m, m), dtype=float)

    # j=1 closure row for u''''(h)
    D4[0, 0] = 16.0
    D4[0, 1] = -9.0
    D4[0, 2] = 8.0/3.0
    D4[0, 3] = -1.0/4.0

    # j=2 standard 5-pt for u''''(2h)
    D4[1, 0] = -4.0
    D4[1, 1] =  6.0
    D4[1, 2] = -4.0
    D4[1, 3] =  1.0

    # interior rows
    for r in range(2, m-2):
        D4[r, r-2] =  1.0
        D4[r, r-1] = -4.0
        D4[r, r  ] =  6.0
        D4[r, r+1] = -4.0
        D4[r, r+2] =  1.0

    # j=N-2 standard row
    r = m - 2
    D4[r, r-2] =  1.0
    D4[r, r-1] = -4.0
    D4[r, r  ] =  6.0
    D4[r, r+1] = -4.0

    # j=N-1 closure row for u''''(1-h)
    r = m - 1
    D4[r, r  ] = 16.0
    D4[r, r-1] = -9.0
    D4[r, r-2] = 8.0/3.0
    D4[r, r-3] = -1.0/4.0

    # scale to 1/h^4
    D4 = D4.tocsr() / (h**4)

    # derivative boundary RHS bases:
    # left closure: (-5*h*du0)/h^4 = -5*du0/h^3
    # right closure: (+5*h*du1)/h^4 = +5*du1/h^3
    b_du0 = np.zeros(m)
    b_du1 = np.zeros(m)
    b_du0[0]  = -5.0 / (h**3)
    b_du1[-1] = +5.0 / (h**3)

    return D4, b_du0, b_du1, h


# ---------------------------
# Manufactured solution (3D clamped, NONZERO derivative BC)
# u = phi(x) sin(pi y) sin(pi z), phi=x(1-x)
# ---------------------------

pi = np.pi

def phi(x):
    return x * (1.0 - x)

def u_exact_3d(X, Y, Z):
    return phi(X) * np.sin(pi*Y) * np.sin(pi*Z)

def rhs_f_3d(X, Y, Z):
    # f = Δ^2 u = 4π^4 u + 8π^2 sin(πy) sin(πz)
    u = u_exact_3d(X, Y, Z)
    return 4.0*(pi**4)*u + 8.0*(pi**2)*np.sin(pi*Y)*np.sin(pi*Z)


# ---------------------------
# Build 3D operator A and boundary RHS
# ---------------------------

def build_A_3d(N):
    """
    A = D4⊗I⊗I + I⊗D4⊗I + I⊗I⊗D4
      + 2R⊗R⊗I + 2R⊗I⊗R + 2I⊗R⊗R
    """
    R, h = laplace_1d_dirichlet_matrix(N)
    D4, b_du0, b_du1, _ = D4_1d_clamped_matrix_and_bases(N)

    m = N - 1
    I = sp.identity(m, format="csr")

    A = (
        kron3(D4, I,  I) +
        kron3(I,  D4, I) +
        kron3(I,  I,  D4) +
        2.0 * kron3(R, R, I) +
        2.0 * kron3(R, I, R) +
        2.0 * kron3(I, R, R)
    ).tocsr()
    #
    # print(np.linalg.cond(A.toarray()))

    return A, h, b_du0, b_du1


def build_boundary_rhs_3d(N, b_du0, b_du1):
    """
    Build b_total such that discrete operator satisfies:
      Δ_h^2 u ≈ A u + b_total
    for the manufactured NONZERO derivative BC.

    Along each 1D line (x or y or z), we have u(endpoints)=0, but du0 and du1 are nonzero,
    and in this manufactured solution du1 = -du0 on every face.
    Hence b_line = (b_du0 - b_du1) * du0.
    """
    h = 1.0 / N
    m = N - 1
    xi = np.linspace(h, 1.0 - h, m)

    bd = (b_du0 - b_du1)        # length m

    sinv = np.sin(pi*xi)
    phiv = phi(xi)

    # x-direction: du0_x(y,z) = sin(pi y) sin(pi z)
    bx = kron3_vec(bd, sinv, sinv)

    # y-direction: du0_y(x,z) = +pi*phi(x)*sin(pi z)
    by = kron3_vec(pi*phiv, bd, sinv)

    # z-direction: du0_z(x,y) = +pi*phi(x)*sin(pi y)
    bz = kron3_vec(pi*phiv, sinv, bd)

    return bx + by + bz


# ---------------------------
# Solve + errors + convergence table
# ---------------------------

def solve_3d_clamped_nonzero_derivative_bc(N):
    if N < 5:
        raise ValueError("N must be >= 5 for this D4 closure stencil.")

    A, h, b_du0, b_du1 = build_A_3d(N)
    m = N - 1

    xi = np.linspace(h, 1.0 - h, m)
    X, Y, Z = np.meshgrid(xi, xi, xi, indexing="ij")

    u_ex = u_exact_3d(X, Y, Z).reshape(-1, order="C")
    f    = rhs_f_3d(X, Y, Z).reshape(-1, order="C")

    b_total = build_boundary_rhs_3d(N, b_du0, b_du1)

    rhs = f - b_total
    if rhs.shape != (A.shape[0],):
        raise RuntimeError(f"RHS shape mismatch: rhs {rhs.shape}, A {A.shape}")

    u_num = spla.spsolve(A, rhs)

    err = u_num - u_ex
    l2 = np.sqrt(np.sum(err**2) * (h**3))
    linf = np.max(np.abs(err))
    return h, l2, linf


def convergence_table(N_list):
    rows = []
    for N in N_list:
        h, e2, einf = solve_3d_clamped_nonzero_derivative_bc(N)
        rows.append((N, h, e2, einf))

    ord_l2 = [np.nan]
    ord_linf = [np.nan]
    for k in range(1, len(rows)):
        _, h1, e1, ef1 = rows[k-1]
        _, h2, e2, ef2 = rows[k]
        ord_l2.append(np.log(e1/e2) / np.log(h1/h2))
        ord_linf.append(np.log(ef1/ef2) / np.log(h1/h2))

    print("   N        h            ||e||_L2        order(L2)      ||e||_Linf       order(Linf)")
    print("-"*98)
    for (N, h, e2, einf), p2, pinf in zip(rows, ord_l2, ord_linf):
        p2s = f"{p2:.4f}" if np.isfinite(p2) else "-"
        pIs = f"{pinf:.4f}" if np.isfinite(pinf) else "-"
        print(f"{N:4d}   {h:11.4e}   {e2:14.6e}     {p2s:>8s}     {einf:14.6e}     {pIs:>8s}")


if __name__ == "__main__":
    # 3D unknowns = (N-1)^3. 直接 spsolve 不要选太大。
    convergence_table([10, 15, 20, 25])

   N        h            ||e||_L2        order(L2)      ||e||_Linf       order(Linf)
--------------------------------------------------------------------------------------------------
  10    1.0000e-01     3.439300e-04            -       1.236464e-03            -
  15    6.6667e-02     1.548077e-04       1.9687       5.432640e-04       2.0283
  20    5.0000e-02     8.744874e-05       1.9853       3.140687e-04       1.9048
  25    4.0000e-02     5.607466e-05       1.9914       1.996941e-04       2.0293
